# SEC EDGAR 파일링에서 주요 데이터를 추출

#### **주요기능:**

| 기능 | 설명 |
|-----|-----|
| fetch_cik_by_ticker() | 티커 심볼로 CIK 번호 조회 | 
| fetch_company_filings() | 회사의 파일링 목록 조회 |
| extract_10k_data() | 10-K에서 섹션, 재무데이터, 리스크 팩터 추출 |
| extract_10q_data() | 10-Q에서 분기 재무데이터 및 MD&A 추출 |
| extract_8k_data() | 8-K에서 공시 이벤트 및 아이템 코드 추출 |
| fetch_13f_filings() | 기관투자자의 13F 파일링 목록 조회 |
| extract_13f_data() | 13F에서 전체 포트폴리오 보유 종목 추출 |
| fetch_13f_filings() | 두 분기 13F 비교 (신규/청산/증감 분석) |
| search_13f_by_stock() | 특정 주식 보유 기관 검색 |
| fetch_institutional_holders() | 주식별 주요 기관 보유자 조회 |
| fetch_def14a_filings() | DEF 14A 파일링 목록 조회 |
| extract_def14a_data() | DEF 14A에서 전체 프록시 데이터 추출 |
| analyze_executive_compensation_trends() | 다년간 임원 보상 트렌드 분석 |
| compare_peer_compensation() | 동종 업계 임원 보상 비교 |
| search_filings() | EDGAR 전문 검색 |

#### **추출 가능한 데이터:**

- **10-K/10-Q 재무 데이터**: 매출, 순이익, 총자산, 부채, 자본, EPS, 현금
- **10-K 섹션**: Business (Item 1), Risk Factors (Item 1A), MD&A (Item 7) 등 전체 섹션
- **8-K 이벤트**: 45개 이상의 8-K 아이템 코드 자동 인식 (임원 변경, M&A, 실적 발표 등)
- **DEF 14A**: Info for shareholders (voting matters)

#### **추출 가능한 13F 데이터:**

- 발행사명, CUSIP, 증권 유형
- 보유 가치 (천 달러 단위)
- 주식 수 또는 원금
- 투자 재량권 (SOLE/SHARED/NONE)
- 의결권 정보
- PUT/CALL 옵션 여부
- 상위 보유 종목 요약

#### **추출 가능한 DEF 14A 데이터:**
| 카테고리 | 세부 항목 | 
|--------|--------|
| 주주총회 정보 | 회의 날짜, 회의 유형, 기준일 | 
| 임원 보상 | 급여, 보너스, 주식/옵션 보상, 비주식 인센티브, 연금, 기타 보상, 총 보상 | 
| CEO Pay Ratio | CEO 대비 중간 직원 급여 비율 | 
| 이사회 | 이사 명단, 나이, 독립성, 위원회, 재임 기간 | 
| 투표 안건 | 제안 번호, 제목, 이사회 권고, 필요 표결 | 
| 주요 주주 | 이름, 지분율, 내부자/기관 구분 | 
| 거버넌스 | Majority voting, Proxy access, Clawback policy 등 |

참조
- https://medium.com/neural-engineer/financial-insights-from-sec-edgar-filings-with-sec-api-io-and-llms-58c479252d7f
- https://medium.com/generative-ai/open-source-finance-research-with-llms-and-sec-edgar-5a0b48e1ab0a
  -  Meta Tools - CIK lookup, tool recommendations
  -  Document Tools - Access and analyze SEC filings
  -  Insider Tools - Extract and analyze insider trading data
  -  Financial Tools - Get key metrics, statements, comparisons
  -  Discovery Tools - Explore available data and concepts
- https://github.com/stefanoamorelli/sec-edgar-mcp
  -  EdgarTools Library (SEC EDGAR REST API)
  -  Direct XBRL Parser (SEC Filing Documents)
- https://www.sec.gov/edgar
  - https://www.sec.gov/edgar/sec-api-documentation
- https://github.com/dgunning/edgartools
- XBRL International: https://www.xbrl.org/

In [ ]:
!pip install requests beautifulsoup4 lxml

SEC EDGAR Filing Crawler
10-K, 10-Q, 8-K 파일링에서 주요 데이터를 추출하는 모듈

추출 가능한 데이터:
- 10-K/10-Q 재무 데이터: 매출, 순이익, 총자산, 부채, 자본, EPS, 현금
- 10-K 섹션: Business (Item 1), Risk Factors (Item 1A), MD&A (Item 7) 등 전체 섹션
- 8-K 이벤트: 45개 이상의 8-K 아이템 코드 자동 인식 (임원 변경, M&A, 실적 발표 등)

sec-api.io

- Advanced XBRL parsing strategies
- Data cleaning and merging for time-series analysis
- LLM prompt engineering for financial analytics
- Error handling and best practices for scalable automation

In [1]:
import re
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass, field
from typing import Optional
import json
import time

In [77]:
@dataclass
class CompanyInfo:
    """회사 기본 정보"""
    cik: str
    name: str
    ticker: Optional[str] = None
    sic: Optional[str] = None
    industry: Optional[str] = None


@dataclass
class FilingMetadata:
    """파일링 메타데이터"""
    accession_number: str
    filing_type: str
    filing_date: str
    period_of_report: Optional[str] = None
    document_url: str = ""


@dataclass
class FinancialData:
    """재무 데이터"""
    revenue: Optional[float] = None
    net_income: Optional[float] = None
    total_assets: Optional[float] = None
    total_liabilities: Optional[float] = None
    stockholders_equity: Optional[float] = None
    eps_basic: Optional[float] = None
    eps_diluted: Optional[float] = None
    operating_income: Optional[float] = None
    cash_and_equivalents: Optional[float] = None


@dataclass
class Filing8KData:
    """8-K 이벤트 데이터"""
    items: list = field(default_factory=list)
    event_descriptions: list = field(default_factory=list)

@dataclass
class Holding13F:
    """13F 개별 보유 종목"""
    issuer_name: str
    title_of_class: str
    cusip: str
    value: float  # USD (thousands)
    shares_or_principal: float
    shares_or_principal_type: str  # SH (shares) or PRN (principal)
    investment_discretion: str  # SOLE, SHARED, NONE
    voting_authority_sole: int = 0
    voting_authority_shared: int = 0
    voting_authority_none: int = 0
    put_call: Optional[str] = None  # PUT, CALL, or None


@dataclass
class Filing13FData:
    """13F 파일링 전체 데이터"""
    filer_name: str = ""
    filer_cik: str = ""
    report_period: str = ""
    filing_date: str = ""
    total_value: float = 0.0  # 총 운용자산 (thousands USD)
    holdings_count: int = 0
    holdings: list = field(default_factory=list)  # List[Holding13F]
    
    # 요약 통계
    top_holdings: list = field(default_factory=list)
    sector_allocation: dict = field(default_factory=dict)

@dataclass
class ExecutiveCompensation:
    """임원 보상 정보"""
    name: str
    title: str
    year: int = 0
    salary: float = 0.0
    bonus: float = 0.0
    stock_awards: float = 0.0
    option_awards: float = 0.0
    non_equity_incentive: float = 0.0
    pension_change: float = 0.0
    other_compensation: float = 0.0
    total: float = 0.0


@dataclass
class DirectorInfo:
    """이사 정보"""
    name: str
    age: Optional[int] = None
    position: str = ""
    committees: list = field(default_factory=list)
    independent: bool = False
    tenure_years: Optional[int] = None
    other_boards: list = field(default_factory=list)
    shares_owned: int = 0
    compensation: float = 0.0


@dataclass 
class ShareholderProposal:
    """주주 제안"""
    proposal_number: int
    title: str
    proponent: str = ""
    description: str = ""
    board_recommendation: str = ""  # FOR, AGAINST, ABSTAIN
    vote_required: str = ""
    prior_year_support: Optional[float] = None


@dataclass
class OwnershipInfo:
    """주요 주주 정보"""
    name: str
    shares: int = 0
    percentage: float = 0.0
    is_insider: bool = False
    is_institution: bool = False


@dataclass
class DEF14AData:
    """DEF 14A Proxy Statement 전체 데이터"""
    company_name: str = ""
    cik: str = ""
    filing_date: str = ""
    meeting_date: str = ""
    meeting_type: str = ""  # Annual, Special
    record_date: str = ""
    
    # 임원 보상
    executive_compensation: list = field(default_factory=list)  # List[ExecutiveCompensation]
    ceo_pay_ratio: Optional[float] = None
    ceo_name: str = ""
    median_employee_pay: Optional[float] = None
    
    # 이사회
    directors: list = field(default_factory=list)  # List[DirectorInfo]
    board_size: int = 0
    independent_directors: int = 0
    board_diversity: dict = field(default_factory=dict)
    
    # 주주 제안 및 투표 안건
    proposals: list = field(default_factory=list)  # List[ShareholderProposal]
    
    # 주요 주주
    major_shareholders: list = field(default_factory=list)  # List[OwnershipInfo]
    insider_ownership_pct: float = 0.0
    institutional_ownership_pct: float = 0.0
    
    # 거버넌스
    governance_highlights: list = field(default_factory=list)
    related_party_transactions: list = field(default_factory=list)
    
    # Say on Pay
    say_on_pay_vote: str = ""  # Annual, Biennial, Triennial
    prior_say_on_pay_support: Optional[float] = None

In [ ]:
class EDGAR13F:
    def fetch_13f_filings(
        self, 
        cik: str, 
        count: int = 10
    ) -> list[FilingMetadata]:
        """
        기관투자자의 13F 파일링 목록 조회
        
        Args:
            cik: CIK 번호
            count: 조회할 파일링 수
        """
        return self.fetch_company_filings(cik, "13F-HR", count)
    
    def extract_13f_data(self, cik: str, accession_number: str) -> Filing13FData:
        """
        13F-HR 파일링에서 포트폴리오 보유 데이터 추출
        
        13F는 $100M 이상 운용 기관투자자가 분기별로 제출하는 보유 주식 공시
        
        Args:
            cik: 기관투자자 CIK
            accession_number: 파일링 접수번호
            
        Returns:
            Filing13FData: 전체 보유 종목 및 요약 정보
        """
        docs = self.fetch_filing_documents(accession_number, cik)
        
        result = Filing13FData(filer_cik=cik)
        
        # 13F 정보 테이블 XML 파일 찾기
        info_table_file = None
        primary_doc = None
        
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "").lower()
            doc_type = item.get("type", "").upper()
            
            # 정보 테이블 (실제 보유 종목 데이터)
            if "infotable" in name or doc_type == "INFORMATION TABLE":
                info_table_file = item.get("name")
            # 메인 13F 문서 (primary_doc.xml)
            elif name.endswith(".xml") and ("13f" in name or "primary" in name):
                primary_doc = item.get("name")
            elif doc_type == "13F-HR" and name.endswith(".xml"):
                primary_doc = item.get("name")
        
        # Primary 문서에서 기본 정보 추출
        if primary_doc:
            self._extract_13f_header(cik, accession_number, primary_doc, result)
        
        # 정보 테이블에서 보유 종목 추출
        if info_table_file:
            self._extract_13f_holdings(cik, accession_number, info_table_file, result)
        else:
            # XML 정보 테이블이 없으면 다른 형식 시도
            for item in docs.get("directory", {}).get("item", []):
                name = item.get("name", "").lower()
                if name.endswith(".xml") and "info" in name:
                    self._extract_13f_holdings(cik, accession_number, item.get("name"), result)
                    break
        
        # 요약 통계 계산
        self._calculate_13f_summary(result)
        
        return result
    
    def _extract_13f_header(
        self, 
        cik: str, 
        accession_number: str, 
        doc_name: str,
        result: Filing13FData
    ):
        """13F 헤더 정보 추출"""
        try:
            content = self.fetch_filing_document_content(cik, accession_number, doc_name)
            soup = BeautifulSoup(content, "xml")
            
            # 제출자 정보
            filer_name = soup.find(["filingManager", "name"])
            if filer_name:
                result.filer_name = filer_name.get_text(strip=True)
            
            # 보고 기간
            period = soup.find(["reportCalendarOrQuarter", "periodOfReport"])
            if period:
                result.report_period = period.get_text(strip=True)
            
            # 제출일
            filing_date = soup.find("signatureDate")
            if filing_date:
                result.filing_date = filing_date.get_text(strip=True)
                
        except Exception as e:
            print(f"13F 헤더 파싱 오류: {e}")
    
    def _extract_13f_holdings(
        self, 
        cik: str, 
        accession_number: str, 
        doc_name: str,
        result: Filing13FData
    ):
        """13F 정보 테이블에서 보유 종목 추출"""
        try:
            content = self.fetch_filing_document_content(cik, accession_number, doc_name)
            soup = BeautifulSoup(content, "xml")
            
            # 다양한 XML 네임스페이스 처리
            # 각 infoTable 엔트리 찾기
            entries = soup.find_all(["infoTable", "infotable", "ns1:infoTable"])
            
            for entry in entries:
                holding = self._parse_13f_entry(entry)
                if holding:
                    result.holdings.append(holding)
            
            result.holdings_count = len(result.holdings)
            result.total_value = sum(h.value for h in result.holdings)
            
        except Exception as e:
            print(f"13F 보유 종목 파싱 오류: {e}")
    
    def _parse_13f_entry(self, entry) -> Optional[Holding13F]:
        """개별 13F 엔트리 파싱"""
        try:
            # 발행사 이름
            issuer = entry.find(["nameOfIssuer", "issuer", "ns1:nameOfIssuer"])
            issuer_name = issuer.get_text(strip=True) if issuer else ""
            
            # 증권 유형
            title = entry.find(["titleOfClass", "title", "ns1:titleOfClass"])
            title_of_class = title.get_text(strip=True) if title else ""
            
            # CUSIP
            cusip = entry.find(["cusip", "ns1:cusip"])
            cusip_val = cusip.get_text(strip=True) if cusip else ""
            
            # 가치 (천 달러 단위)
            value = entry.find(["value", "ns1:value"])
            value_thousands = float(value.get_text(strip=True)) if value else 0.0
            
            # 주식 수 또는 원금
            shares_elem = entry.find(["shrsOrPrnAmt", "ns1:shrsOrPrnAmt"])
            shares = 0.0
            shares_type = "SH"
            
            if shares_elem:
                amt = shares_elem.find(["sshPrnamt", "ns1:sshPrnamt"])
                if amt:
                    shares = float(amt.get_text(strip=True))
                type_elem = shares_elem.find(["sshPrnamtType", "ns1:sshPrnamtType"])
                if type_elem:
                    shares_type = type_elem.get_text(strip=True)
            
            # 투자 재량권
            discretion = entry.find(["investmentDiscretion", "ns1:investmentDiscretion"])
            discretion_val = discretion.get_text(strip=True) if discretion else "SOLE"
            
            # 의결권
            voting = entry.find(["votingAuthority", "ns1:votingAuthority"])
            vote_sole = 0
            vote_shared = 0
            vote_none = 0
            
            if voting:
                sole = voting.find(["Sole", "sole", "ns1:Sole"])
                if sole:
                    vote_sole = int(sole.get_text(strip=True) or 0)
                shared = voting.find(["Shared", "shared", "ns1:Shared"])
                if shared:
                    vote_shared = int(shared.get_text(strip=True) or 0)
                none_elem = voting.find(["None", "none", "ns1:None"])
                if none_elem:
                    vote_none = int(none_elem.get_text(strip=True) or 0)
            
            # PUT/CALL 옵션
            put_call = entry.find(["putCall", "ns1:putCall"])
            put_call_val = put_call.get_text(strip=True) if put_call else None
            
            return Holding13F(
                issuer_name=issuer_name,
                title_of_class=title_of_class,
                cusip=cusip_val,
                value=value_thousands,
                shares_or_principal=shares,
                shares_or_principal_type=shares_type,
                investment_discretion=discretion_val,
                voting_authority_sole=vote_sole,
                voting_authority_shared=vote_shared,
                voting_authority_none=vote_none,
                put_call=put_call_val
            )
            
        except Exception as e:
            print(f"13F 엔트리 파싱 오류: {e}")
            return None
    
    def _calculate_13f_summary(self, result: Filing13FData):
        """13F 요약 통계 계산"""
        if not result.holdings:
            return
        
        # 상위 보유 종목 (가치 기준)
        sorted_holdings = sorted(
            result.holdings, 
            key=lambda x: x.value, 
            reverse=True
        )
        
        # 상위 10개 종목
        result.top_holdings = [
            {
                "issuer": h.issuer_name,
                "title": h.title_of_class,
                "cusip": h.cusip,
                "value_thousands": h.value,
                "value_millions": h.value / 1000,
                "shares": h.shares_or_principal,
                "percentage": (h.value / result.total_value * 100) if result.total_value > 0 else 0
            }
            for h in sorted_holdings[:10]
        ]
    
    def compare_13f_filings(
        self, 
        cik: str, 
        accession_current: str, 
        accession_previous: str
    ) -> dict:
        """
        두 분기의 13F 파일링 비교 (포트폴리오 변화 분석)
        
        Args:
            cik: 기관투자자 CIK
            accession_current: 현재 분기 파일링
            accession_previous: 이전 분기 파일링
            
        Returns:
            새로운 포지션, 청산 포지션, 증가/감소 포지션
        """
        current = self.extract_13f_data(cik, accession_current)
        previous = self.extract_13f_data(cik, accession_previous)
        
        # CUSIP 기준으로 매핑
        current_map = {h.cusip: h for h in current.holdings}
        previous_map = {h.cusip: h for h in previous.holdings}
        
        current_cusips = set(current_map.keys())
        previous_cusips = set(previous_map.keys())
        
        # 새로운 포지션
        new_positions = []
        for cusip in (current_cusips - previous_cusips):
            h = current_map[cusip]
            new_positions.append({
                "issuer": h.issuer_name,
                "cusip": cusip,
                "value_thousands": h.value,
                "shares": h.shares_or_principal
            })
        
        # 청산된 포지션
        closed_positions = []
        for cusip in (previous_cusips - current_cusips):
            h = previous_map[cusip]
            closed_positions.append({
                "issuer": h.issuer_name,
                "cusip": cusip,
                "previous_value_thousands": h.value,
                "previous_shares": h.shares_or_principal
            })
        
        # 변동된 포지션 (공통)
        changed_positions = []
        for cusip in (current_cusips & previous_cusips):
            curr = current_map[cusip]
            prev = previous_map[cusip]
            
            share_change = curr.shares_or_principal - prev.shares_or_principal
            value_change = curr.value - prev.value
            
            if share_change != 0:
                changed_positions.append({
                    "issuer": curr.issuer_name,
                    "cusip": cusip,
                    "current_shares": curr.shares_or_principal,
                    "previous_shares": prev.shares_or_principal,
                    "share_change": share_change,
                    "share_change_pct": (share_change / prev.shares_or_principal * 100) 
                                        if prev.shares_or_principal > 0 else 0,
                    "current_value_thousands": curr.value,
                    "previous_value_thousands": prev.value,
                    "value_change_thousands": value_change,
                })
        
        # 변동률 기준 정렬
        changed_positions.sort(key=lambda x: abs(x.get("share_change_pct", 0)), reverse=True)
        
        return {
            "current_period": current.report_period,
            "previous_period": previous.report_period,
            "current_total_value": current.total_value,
            "previous_total_value": previous.total_value,
            "total_value_change": current.total_value - previous.total_value,
            "current_holdings_count": current.holdings_count,
            "previous_holdings_count": previous.holdings_count,
            "new_positions": sorted(new_positions, key=lambda x: x["value_thousands"], reverse=True),
            "closed_positions": sorted(closed_positions, key=lambda x: x["previous_value_thousands"], reverse=True),
            "increased_positions": [p for p in changed_positions if p["share_change"] > 0][:20],
            "decreased_positions": [p for p in changed_positions if p["share_change"] < 0][:20],
        }
    
    def search_13f_by_stock(
        self,
        cusip: str = None,
        ticker: str = None,
        min_value: float = None,
        quarter: str = None
    ) -> list[dict]:
        """
        특정 주식을 보유한 기관투자자 검색
        
        Note: 이 기능은 EDGAR 전문 검색 API의 제한으로 
              완전한 구현이 어려울 수 있음
        """
        # CUSIP 또는 티커로 검색
        query = cusip or ticker
        if not query:
            return []
        
        results = self.search_filings(
            query=query,
            filing_types=["13F-HR"],
            size=20
        )
        
        return results
    
    def fetch_institutional_holders(
        self,
        ticker: str,
        top_n: int = 20
    ) -> list[dict]:
        """
        특정 주식의 주요 기관 보유자 조회
        
        Note: SEC EDGAR는 주식별 검색을 직접 지원하지 않아
              외부 데이터 소스 또는 자체 인덱싱이 필요할 수 있음
        """
        # 기본적인 검색 시도
        results = self.search_filings(
            query=ticker,
            filing_types=["13F-HR"],
            start_date="2024-01-01",
            size=50
        )
        
        holders = []
        seen_ciks = set()
        
        for r in results:
            cik = r.get("cik", "")
            if cik not in seen_ciks:
                seen_ciks.add(cik)
                holders.append({
                    "institution": r.get("company_name", ""),
                    "cik": cik,
                    "latest_filing": r.get("filing_date", ""),
                    "accession": r.get("accession_number", "")
                })
        
        return holders[:top_n]

In [ ]:
class EDGARDEF14A:
    
    # ========================================
    # DEF 14A (Proxy Statement) 파싱
    # ========================================
    
    def fetch_def14a_filings(
        self, 
        cik: str, 
        count: int = 10
    ) -> list[FilingMetadata]:
        """
        회사의 DEF 14A (Proxy Statement) 파일링 목록 조회
        
        Args:
            cik: CIK 번호
            count: 조회할 파일링 수
        """
        return self.fetch_company_filings(cik, "DEF 14A", count)
    
    def extract_def14a_data(self, cik: str, accession_number: str) -> DEF14AData:
        """
        DEF 14A Proxy Statement에서 주요 데이터 추출
        
        DEF 14A는 주주총회 위임장 설명서로 다음 정보를 포함:
        - 임원 보상 (Executive Compensation)
        - 이사회 구성 및 거버넌스
        - 주주 제안 및 투표 안건
        - 주요 주주 지분 현황
        - CEO Pay Ratio
        - Related Party Transactions
        
        Args:
            cik: 회사 CIK
            accession_number: 파일링 접수번호
            
        Returns:
            DEF14AData: 추출된 프록시 데이터
        """
        docs = self.fetch_filing_documents(accession_number, cik)
        result = DEF14AData(cik=cik)
        
        # 메인 DEF 14A 문서 찾기
        main_doc = None
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "").lower()
            doc_type = item.get("type", "").upper()
            
            if doc_type == "DEF 14A" or "def14a" in name or "proxy" in name:
                if name.endswith(".htm") or name.endswith(".html"):
                    main_doc = item.get("name")
                    break
        
        if not main_doc:
            # 가장 큰 htm 파일 선택
            htm_files = [
                item for item in docs.get("directory", {}).get("item", [])
                if item.get("name", "").endswith((".htm", ".html"))
            ]
            if htm_files:
                main_doc = max(htm_files, key=lambda x: int(x.get("size", 0))).get("name")
        
        if main_doc:
            content = self.get_filing_document_content(cik, accession_number, main_doc)
            soup = BeautifulSoup(content, "html.parser")
            
            # 스크립트/스타일 제거
            for tag in soup(["script", "style"]):
                tag.decompose()
            
            text = soup.get_text(separator="\n")
            
            # 각 섹션 추출
            self._extract_def14a_meeting_info(text, result)
            self._extract_executive_compensation(soup, text, result)
            self._extract_director_info(soup, text, result)
            self._extract_proposals(text, result)
            self._extract_ownership_info(soup, text, result)
            self._extract_ceo_pay_ratio(text, result)
            self._extract_governance_info(text, result)
        
        return result
    
    def _extract_def14a_meeting_info(self, text: str, result: DEF14AData):
        """주주총회 기본 정보 추출"""
        
        # 회의 날짜
        meeting_patterns = [
            r"(?:annual|special)\s+meeting.*?(?:will be held|to be held|held)\s+(?:on\s+)?([A-Z][a-z]+\s+\d{1,2},?\s+\d{4})",
            r"meeting\s+date[:\s]+([A-Z][a-z]+\s+\d{1,2},?\s+\d{4})",
            r"(?:on|dated?)\s+([A-Z][a-z]+\s+\d{1,2},?\s+\d{4}).*?(?:annual|special)\s+meeting",
        ]
        
        for pattern in meeting_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                result.meeting_date = match.group(1).strip()
                break
        
        # 회의 유형
        if re.search(r"annual\s+meeting", text, re.IGNORECASE):
            result.meeting_type = "Annual"
        elif re.search(r"special\s+meeting", text, re.IGNORECASE):
            result.meeting_type = "Special"
        
        # Record Date
        record_pattern = r"record\s+date[:\s]+([A-Z][a-z]+\s+\d{1,2},?\s+\d{4})"
        match = re.search(record_pattern, text, re.IGNORECASE)
        if match:
            result.record_date = match.group(1).strip()
    
    def _extract_executive_compensation(
        self, 
        soup: BeautifulSoup, 
        text: str, 
        result: DEF14AData
    ):
        """임원 보상 테이블 추출"""
        
        # Summary Compensation Table 찾기
        tables = soup.find_all("table")
        comp_table = None
        
        for table in tables:
            table_text = table.get_text().lower()
            if "summary compensation" in table_text or \
               ("name" in table_text and "salary" in table_text and "total" in table_text):
                comp_table = table
                break
        
        if comp_table:
            rows = comp_table.find_all("tr")
            header_found = False
            col_indices = {}
            
            for row in rows:
                cells = row.find_all(["td", "th"])
                cell_texts = [c.get_text(strip=True).lower() for c in cells]
                
                # 헤더 행 찾기
                if not header_found:
                    if "name" in cell_texts or "salary" in cell_texts:
                        for i, t in enumerate(cell_texts):
                            if "name" in t:
                                col_indices["name"] = i
                            elif "year" in t:
                                col_indices["year"] = i
                            elif "salary" in t:
                                col_indices["salary"] = i
                            elif "bonus" in t:
                                col_indices["bonus"] = i
                            elif "stock" in t and "award" in t:
                                col_indices["stock_awards"] = i
                            elif "option" in t:
                                col_indices["option_awards"] = i
                            elif "non-equity" in t or "incentive" in t:
                                col_indices["non_equity"] = i
                            elif "pension" in t or "deferred" in t:
                                col_indices["pension"] = i
                            elif "other" in t:
                                col_indices["other"] = i
                            elif "total" in t:
                                col_indices["total"] = i
                        header_found = True
                        continue
                
                # 데이터 행 파싱
                if header_found and len(cells) >= 3:
                    try:
                        name = cells[col_indices.get("name", 0)].get_text(strip=True)
                        
                        # 빈 행이나 헤더 반복 건너뛰기
                        if not name or name.lower() in ["name", "principal position", ""]:
                            continue
                        
                        exec_comp = ExecutiveCompensation(
                            name=name,
                            title=self._extract_title_from_name(name)
                        )
                        
                        # 각 필드 추출
                        if "year" in col_indices:
                            year_text = cells[col_indices["year"]].get_text(strip=True)
                            exec_comp.year = self._parse_int(year_text)
                        
                        if "salary" in col_indices:
                            exec_comp.salary = self._parse_currency(
                                cells[col_indices["salary"]].get_text(strip=True)
                            )
                        
                        if "bonus" in col_indices:
                            exec_comp.bonus = self._parse_currency(
                                cells[col_indices["bonus"]].get_text(strip=True)
                            )
                        
                        if "stock_awards" in col_indices:
                            exec_comp.stock_awards = self._parse_currency(
                                cells[col_indices["stock_awards"]].get_text(strip=True)
                            )
                        
                        if "option_awards" in col_indices:
                            exec_comp.option_awards = self._parse_currency(
                                cells[col_indices["option_awards"]].get_text(strip=True)
                            )
                        
                        if "non_equity" in col_indices:
                            exec_comp.non_equity_incentive = self._parse_currency(
                                cells[col_indices["non_equity"]].get_text(strip=True)
                            )
                        
                        if "pension" in col_indices:
                            exec_comp.pension_change = self._parse_currency(
                                cells[col_indices["pension"]].get_text(strip=True)
                            )
                        
                        if "other" in col_indices:
                            exec_comp.other_compensation = self._parse_currency(
                                cells[col_indices["other"]].get_text(strip=True)
                            )
                        
                        if "total" in col_indices:
                            exec_comp.total = self._parse_currency(
                                cells[col_indices["total"]].get_text(strip=True)
                            )
                        
                        # 총액이 없으면 계산
                        if exec_comp.total == 0:
                            exec_comp.total = (
                                exec_comp.salary + exec_comp.bonus + 
                                exec_comp.stock_awards + exec_comp.option_awards +
                                exec_comp.non_equity_incentive + exec_comp.pension_change +
                                exec_comp.other_compensation
                            )
                        
                        if exec_comp.total > 0:
                            result.executive_compensation.append(exec_comp)
                            
                    except Exception as e:
                        continue
        
        # CEO 식별
        if result.executive_compensation:
            for exec_comp in result.executive_compensation:
                title_lower = exec_comp.title.lower()
                if "ceo" in title_lower or "chief executive" in title_lower:
                    result.ceo_name = exec_comp.name
                    break
            
            # CEO 타이틀이 없으면 첫 번째 (보통 CEO)
            if not result.ceo_name:
                result.ceo_name = result.executive_compensation[0].name
    
    def _extract_title_from_name(self, name_cell: str) -> str:
        """이름 셀에서 직책 추출"""
        # 줄바꿈이나 쉼표로 구분된 경우
        parts = re.split(r'[\n,]', name_cell)
        if len(parts) > 1:
            return parts[1].strip()
        return ""
    
    def _extract_director_info(
        self, 
        soup: BeautifulSoup, 
        text: str, 
        result: DEF14AData
    ):
        """이사회 정보 추출"""
        
        # Director 테이블 또는 섹션 찾기
        directors = []
        
        # "Director" 또는 "Board" 섹션에서 이름 추출
        director_pattern = r"(?:^|\n)([A-Z][a-z]+(?:\s+[A-Z]\.?\s+)?[A-Z][a-z]+)[,\s]+(?:age\s+)?(\d{2,3})?[,\s]*(?:has been|has served|is|serves as|director)"
        
        matches = re.findall(director_pattern, text, re.MULTILINE)
        
        seen_names = set()
        for match in matches:
            name = match[0].strip()
            if name not in seen_names and len(name) > 5:
                seen_names.add(name)
                
                age = int(match[1]) if match[1] else None
                
                director = DirectorInfo(
                    name=name,
                    age=age
                )
                
                # Independent 여부 확인
                name_context = text[text.find(name):text.find(name) + 500]
                if re.search(r"independent", name_context, re.IGNORECASE):
                    director.independent = True
                
                directors.append(director)
        
        # 테이블에서도 추출 시도
        tables = soup.find_all("table")
        for table in tables:
            table_text = table.get_text().lower()
            if "director" in table_text and ("age" in table_text or "since" in table_text):
                rows = table.find_all("tr")
                for row in rows[1:]:  # 헤더 건너뛰기
                    cells = row.find_all(["td", "th"])
                    if len(cells) >= 2:
                        name = cells[0].get_text(strip=True)
                        if name and len(name) > 3 and name not in seen_names:
                            seen_names.add(name)
                            
                            director = DirectorInfo(name=name)
                            
                            # 나이 찾기
                            for cell in cells[1:]:
                                cell_text = cell.get_text(strip=True)
                                age_match = re.match(r'^(\d{2,3})$', cell_text)
                                if age_match:
                                    director.age = int(age_match.group(1))
                                    break
                            
                            directors.append(director)
        
        result.directors = directors[:20]  # 최대 20명
        result.board_size = len(result.directors)
        result.independent_directors = sum(1 for d in result.directors if d.independent)
    
    def _extract_proposals(self, text: str, result: DEF14AData):
        """주주 제안 및 투표 안건 추출"""
        
        proposals = []
        
        # Proposal 패턴
        proposal_patterns = [
            r"Proposal\s+(?:No\.?\s*)?(\d+)[:\s]+(.+?)(?=Proposal\s+(?:No\.?\s*)?\d+|$)",
            r"Item\s+(\d+)[:\s]+(.+?)(?=Item\s+\d+|$)",
        ]
        
        for pattern in proposal_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE | re.DOTALL)
            if matches:
                for num, title in matches:
                    # 제목 정리
                    title_clean = title.strip()[:200]
                    title_clean = re.sub(r'\s+', ' ', title_clean)
                    
                    if len(title_clean) < 10:
                        continue
                    
                    proposal = ShareholderProposal(
                        proposal_number=int(num),
                        title=title_clean
                    )
                    
                    # Board Recommendation 찾기
                    rec_pattern = rf"Proposal\s+(?:No\.?\s*)?{num}.*?(?:board|our)\s+recommend(?:s|ation)[:\s]+(\w+)"
                    rec_match = re.search(rec_pattern, text, re.IGNORECASE | re.DOTALL)
                    if rec_match:
                        rec = rec_match.group(1).upper()
                        if rec in ["FOR", "AGAINST", "ABSTAIN"]:
                            proposal.board_recommendation = rec
                    
                    proposals.append(proposal)
                break
        
        # 일반적인 안건 유형 감지
        common_proposals = [
            (r"election\s+of\s+directors", "Election of Directors"),
            (r"ratif(?:y|ication)\s+.*?(?:independent\s+)?auditor", "Ratification of Auditors"),
            (r"advisory\s+(?:vote|approval)\s+.*?(?:executive\s+)?compensation", "Say on Pay"),
            (r"say.?on.?pay", "Say on Pay"),
            (r"approve\s+.*?(?:equity|incentive|stock)\s+plan", "Equity Compensation Plan"),
            (r"amend(?:ment)?\s+.*?(?:certificate|charter|bylaws)", "Charter/Bylaws Amendment"),
        ]
        
        for pattern, title in common_proposals:
            if re.search(pattern, text, re.IGNORECASE):
                # 이미 추가되지 않은 경우에만
                if not any(title.lower() in p.title.lower() for p in proposals):
                    proposals.append(ShareholderProposal(
                        proposal_number=len(proposals) + 1,
                        title=title,
                        board_recommendation="FOR"
                    ))
        
        result.proposals = proposals
    
    def _extract_ownership_info(
        self, 
        soup: BeautifulSoup, 
        text: str, 
        result: DEF14AData
    ):
        """주요 주주 정보 추출"""
        
        shareholders = []
        
        # Beneficial Ownership 테이블 찾기
        tables = soup.find_all("table")
        
        for table in tables:
            table_text = table.get_text().lower()
            if "beneficial" in table_text and "percent" in table_text:
                rows = table.find_all("tr")
                
                for row in rows[1:]:
                    cells = row.find_all(["td", "th"])
                    if len(cells) >= 2:
                        name = cells[0].get_text(strip=True)
                        
                        if not name or len(name) < 3:
                            continue
                        
                        # 숫자가 아닌 첫 번째 셀 = 이름
                        if re.match(r'^[\d,.\s%]+$', name):
                            continue
                        
                        owner = OwnershipInfo(name=name)
                        
                        # 주식 수와 퍼센트 추출
                        for cell in cells[1:]:
                            cell_text = cell.get_text(strip=True)
                            
                            # 퍼센트
                            pct_match = re.search(r'([\d.]+)\s*%', cell_text)
                            if pct_match and owner.percentage == 0:
                                owner.percentage = float(pct_match.group(1))
                            
                            # 주식 수
                            shares_match = re.match(r'^([\d,]+)$', cell_text.replace(',', ''))
                            if shares_match and owner.shares == 0:
                                owner.shares = self._parse_int(cell_text)
                        
                        # 기관 vs 내부자 분류
                        name_lower = name.lower()
                        if any(kw in name_lower for kw in ["fund", "capital", "management", 
                                                            "partners", "advisors", "investment",
                                                            "blackrock", "vanguard", "state street"]):
                            owner.is_institution = True
                        else:
                            owner.is_insider = True
                        
                        if owner.percentage > 0 or owner.shares > 0:
                            shareholders.append(owner)
                
                break
        
        result.major_shareholders = shareholders[:20]
        
        # 내부자/기관 소유 합계
        result.insider_ownership_pct = sum(
            s.percentage for s in shareholders if s.is_insider
        )
        result.institutional_ownership_pct = sum(
            s.percentage for s in shareholders if s.is_institution
        )
    
    def _extract_ceo_pay_ratio(self, text: str, result: DEF14AData):
        """CEO Pay Ratio 추출"""
        
        # Pay Ratio 패턴
        ratio_patterns = [
            r"(?:ceo|chief executive).*?pay\s+ratio.*?(\d+)\s*(?:to|:)\s*1",
            r"pay\s+ratio.*?(\d+)\s*(?:to|:)\s*1",
            r"ratio.*?(?:ceo|chief executive).*?(\d+)\s*(?:to|:)\s*1",
            r"(\d+)\s*(?:to|:)\s*1\s*(?:ratio|pay ratio)",
        ]
        
        for pattern in ratio_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                result.ceo_pay_ratio = float(match.group(1))
                break
        
        # Median Employee Pay
        median_patterns = [
            r"median\s+(?:annual\s+)?(?:total\s+)?compensation.*?\$?([\d,]+)",
            r"median\s+employee.*?\$?([\d,]+)",
        ]
        
        for pattern in median_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                result.median_employee_pay = self._parse_currency(match.group(1))
                break
    
    def _extract_governance_info(self, text: str, result: DEF14AData):
        """거버넌스 정보 추출"""
        
        governance_items = []
        
        # 일반적인 거버넌스 특징 감지
        governance_patterns = [
            (r"majority\s+voting", "Majority voting for directors"),
            (r"proxy\s+access", "Proxy access"),
            (r"annual\s+election.*?directors", "Annual election of directors"),
            (r"no\s+poison\s+pill", "No poison pill"),
            (r"independent\s+(?:board\s+)?chair", "Independent board chair"),
            (r"lead\s+independent\s+director", "Lead independent director"),
            (r"clawback\s+polic", "Clawback policy"),
            (r"stock\s+ownership\s+guideline", "Stock ownership guidelines"),
            (r"no\s+hedging", "Anti-hedging policy"),
            (r"no\s+pledging", "Anti-pledging policy"),
            (r"overboarding\s+polic", "Director overboarding policy"),
            (r"board\s+diversity", "Board diversity policy"),
            (r"esg|sustainability\s+committee", "ESG/Sustainability focus"),
        ]
        
        for pattern, description in governance_patterns:
            if re.search(pattern, text, re.IGNORECASE):
                governance_items.append(description)
        
        result.governance_highlights = governance_items
        
        # Say on Pay 빈도
        if re.search(r"say.?on.?pay.*?every\s+year|annual.*?say.?on.?pay", text, re.IGNORECASE):
            result.say_on_pay_vote = "Annual"
        elif re.search(r"say.?on.?pay.*?every\s+two\s+years|biennial", text, re.IGNORECASE):
            result.say_on_pay_vote = "Biennial"
        elif re.search(r"say.?on.?pay.*?every\s+three\s+years|triennial", text, re.IGNORECASE):
            result.say_on_pay_vote = "Triennial"
        
        # 작년 Say on Pay 지지율
        support_pattern = r"(?:last|prior|previous)\s+year.*?say.?on.?pay.*?(\d+(?:\.\d+)?)\s*%"
        match = re.search(support_pattern, text, re.IGNORECASE)
        if match:
            result.prior_say_on_pay_support = float(match.group(1))
        
        # Related Party Transactions
        rpt_pattern = r"related\s+(?:party|person)\s+transaction.*?(?:\$|USD)?\s*([\d,]+(?:\.\d+)?)\s*(?:million|M)?"
        matches = re.findall(rpt_pattern, text, re.IGNORECASE)
        for m in matches[:5]:
            result.related_party_transactions.append(f"${m}")
    
    def _parse_currency(self, text: str) -> float:
        """통화 문자열을 숫자로 변환"""
        if not text:
            return 0.0
        
        # 특수 문자 제거
        text = text.replace("$", "").replace(",", "").replace("—", "0").replace("-", "0")
        text = text.strip()
        
        if not text:
            return 0.0
        
        try:
            return float(text)
        except ValueError:
            return 0.0
    
    def _parse_int(self, text: str) -> int:
        """정수 문자열을 숫자로 변환"""
        if not text:
            return 0
        
        text = text.replace(",", "").replace("$", "").strip()
        
        try:
            return int(float(text))
        except ValueError:
            return 0
    
    def analyze_executive_compensation_trends(
        self,
        cik: str,
        years: int = 3
    ) -> dict:
        """
        여러 연도의 임원 보상 트렌드 분석
        
        Args:
            cik: 회사 CIK
            years: 분석할 연도 수
            
        Returns:
            연도별 임원 보상 및 변화율
        """
        filings = self.fetch_def14a_filings(cik, count=years)
        
        yearly_data = []
        
        for filing in filings:
            data = self.extract_def14a_data(cik, filing.accession_number)
            
            if data.executive_compensation:
                ceo_comp = None
                total_named_exec_comp = 0
                
                for exec_comp in data.executive_compensation:
                    total_named_exec_comp += exec_comp.total
                    
                    if exec_comp.name == data.ceo_name:
                        ceo_comp = exec_comp.total
                
                yearly_data.append({
                    "filing_date": filing.filing_date,
                    "ceo_name": data.ceo_name,
                    "ceo_total_compensation": ceo_comp,
                    "ceo_pay_ratio": data.ceo_pay_ratio,
                    "total_named_exec_compensation": total_named_exec_comp,
                    "number_of_named_executives": len(data.executive_compensation),
                    "median_employee_pay": data.median_employee_pay,
                })
        
        # YoY 변화율 계산
        for i in range(len(yearly_data) - 1):
            current = yearly_data[i]
            previous = yearly_data[i + 1]
            
            if current.get("ceo_total_compensation") and previous.get("ceo_total_compensation"):
                change = (current["ceo_total_compensation"] - previous["ceo_total_compensation"]) / previous["ceo_total_compensation"] * 100
                current["ceo_comp_yoy_change_pct"] = round(change, 2)
        
        return {
            "company_cik": cik,
            "years_analyzed": len(yearly_data),
            "compensation_by_year": yearly_data
        }
    
    def compare_peer_compensation(
        self,
        ciks: list[str]
    ) -> list[dict]:
        """
        여러 회사의 임원 보상 비교
        
        Args:
            ciks: 비교할 회사들의 CIK 리스트
            
        Returns:
            회사별 임원 보상 비교 데이터
        """
        comparison = []
        
        for cik in ciks:
            try:
                filings = self.fetch_def14a_filings(cik, count=1)
                if filings:
                    data = self.extract_def14a_data(cik, filings[0].accession_number)
                    
                    ceo_comp = None
                    for exec_comp in data.executive_compensation:
                        if exec_comp.name == data.ceo_name:
                            ceo_comp = exec_comp
                            break
                    
                    if not ceo_comp and data.executive_compensation:
                        ceo_comp = data.executive_compensation[0]
                    
                    comparison.append({
                        "cik": cik,
                        "company_name": data.company_name,
                        "ceo_name": data.ceo_name,
                        "ceo_total_compensation": ceo_comp.total if ceo_comp else 0,
                        "ceo_salary": ceo_comp.salary if ceo_comp else 0,
                        "ceo_stock_awards": ceo_comp.stock_awards if ceo_comp else 0,
                        "ceo_pay_ratio": data.ceo_pay_ratio,
                        "board_size": data.board_size,
                        "independent_directors": data.independent_directors,
                        "filing_date": filings[0].filing_date,
                    })
            except Exception as e:
                print(f"Error processing CIK {cik}: {e}")
                continue
        
        # CEO 보상 기준 정렬
        comparison.sort(key=lambda x: x.get("ceo_total_compensation", 0), reverse=True)
        
        return comparison

In [69]:
class EDGARCrawler:
    """SEC EDGAR 파일링 추출기"""
    
    BASE_URL = "https://www.sec.gov"
    EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"
    COMPANY_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
    
    # 8-K 아이템 코드 매핑
    ITEM_8K_MAPPING = {
        "1.01": "Entry into a Material Definitive Agreement",
        "1.02": "Termination of a Material Definitive Agreement",
        "1.03": "Bankruptcy or Receivership",
        "1.04": "Mine Safety - Reporting of Shutdowns and Patterns of Violations",
        "2.01": "Completion of Acquisition or Disposition of Assets",
        "2.02": "Results of Operations and Financial Condition",
        "2.03": "Creation of a Direct Financial Obligation",
        "2.04": "Triggering Events That Accelerate or Increase Obligation",
        "2.05": "Costs Associated with Exit or Disposal Activities",
        "2.06": "Material Impairments",
        "3.01": "Notice of Delisting or Transfer",
        "3.02": "Unregistered Sales of Equity Securities",
        "3.03": "Material Modification to Rights of Security Holders",
        "4.01": "Changes in Registrant's Certifying Accountant",
        "4.02": "Non-Reliance on Previously Issued Financial Statements",
        "5.01": "Changes in Control of Registrant",
        "5.02": "Departure/Election of Directors or Officers",
        "5.03": "Amendments to Articles of Incorporation or Bylaws",
        "5.04": "Temporary Suspension of Trading Under Employee Benefit Plans",
        "5.05": "Amendment to Registrant's Code of Ethics",
        "5.06": "Change in Shell Company Status",
        "5.07": "Submission of Matters to a Vote of Security Holders",
        "5.08": "Shareholder Nominations",
        "6.01": "ABS Informational and Computational Material",
        "6.02": "Change of Servicer or Trustee",
        "6.03": "Change in Credit Enhancement or External Support",
        "6.04": "Failure to Make a Required Distribution",
        "6.05": "Securities Act Updating Disclosure",
        "7.01": "Regulation FD Disclosure",
        "8.01": "Other Events",
        "9.01": "Financial Statements and Exhibits",
    }
    
    def __init__(self, user_agent: str = "Sayouzone sjkim@sayouzone.com"):
        """
        Args:
            user_agent: SEC에서 요구하는 User-Agent 헤더 (회사명 + 이메일)
        """
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": user_agent,
            "Accept-Encoding": "gzip, deflate",
        })
        self._ticker_to_cik = None
    
    def _rate_limit(self):
        """SEC 요청 제한 준수 (초당 10회 이하)"""
        time.sleep(0.1)
    
    def fetch_cik_by_ticker(self, ticker: str) -> Optional[str]:
        """티커로 CIK 번호 조회"""
        if self._ticker_to_cik is None:
            self._rate_limit()
            resp = self.session.get(self.COMPANY_TICKERS_URL)
            resp.raise_for_status()
            data = resp.json()
            self._ticker_to_cik = {
                v["ticker"]: str(v["cik_str"]).zfill(10)
                for v in data.values()
            }
        return self._ticker_to_cik.get(ticker.upper())
    
    def fetch_company_filings(
        self, 
        cik: str, 
        filing_type: str = "10-K", 
        count: int = 10
    ) -> list[FilingMetadata]:
        """
        회사의 파일링 목록 조회
        
        Args:
            cik: CIK 번호 (앞에 0 채워서 10자리)
            filing_type: 파일링 유형 (10-K, 10-Q, 8-K), Form 13F (Data Sets)
            count: 조회할 파일링 수
        """
        cik = cik.zfill(10)
        url = f"{self.BASE_URL}/cgi-bin/browse-edgar"
        params = {
            "action": "getcompany",
            "CIK": cik,
            "type": filing_type,
            "dateb": "",
            "owner": "include",
            "count": count,
            "output": "atom",
        }
        
        self._rate_limit()
        resp = self.session.get(url, params=params)
        resp.raise_for_status()
        
        soup = BeautifulSoup(resp.content, "xml")
        filings = []
        
        for entry in soup.find_all("entry"):
            accession = entry.find("accession-number")
            if not accession:
                continue
                
            filing_date = entry.find("filing-date")
            filing_href = entry.find("filing-href")
            #link_href = entry.find("link")["href"] # filing-href와 동일
            
            metadata = FilingMetadata(
                accession_number=accession.text.strip(),
                filing_type=filing_type,
                filing_date=filing_date.text.strip() if filing_date else "",
                document_url=filing_href.text.strip() if filing_href else "",
            )
            filings.append(metadata)
        
        return filings
    
    def fetch_filing_documents(self, accession_number: str, cik: str) -> dict:
        """파일링의 문서 목록 조회"""
        cik = cik.zfill(10)
        accession_clean = accession_number.replace("-", "")
        
        index_url = f"{self.BASE_URL}/Archives/edgar/data/{cik}/{accession_clean}/index.json"
        
        self._rate_limit()
        resp = self.session.get(index_url)
        resp.raise_for_status()
        
        return resp.json()
    
    def fetch_filing_document_content(
        self, 
        cik: str, 
        accession_number: str, 
        document_name: str
    ) -> str:
        """특정 파일링 문서 내용 가져오기"""
        cik = cik.zfill(10)
        accession_clean = accession_number.replace("-", "")
        
        doc_url = f"{self.BASE_URL}/Archives/edgar/data/{cik}/{accession_clean}/{document_name}"
        
        self._rate_limit()
        resp = self.session.get(doc_url)
        resp.raise_for_status()
        
        return resp.text
    
    def fetch_document_content(
        self, 
        url: str
    ) -> str:
        """특정 URL에서 문서 내용 가져오기"""
        
        self._rate_limit()
        resp = self.session.get(url)
        resp.raise_for_status()
        
        return resp.content
    
    def extract_10k_data(self, cik: str, accession_number: str) -> dict:
        """
        10-K 파일링에서 주요 데이터 추출
        
        Returns:
            섹션별 텍스트와 재무 데이터
        """
        docs = self.fetch_filing_documents(accession_number, cik)
        
        # 메인 10-K 문서 찾기
        main_doc = None
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if name.endswith(".htm") and "10-k" in name.lower():
                main_doc = name
                break
            elif name.endswith(".htm") and item.get("type") == "10-K":
                main_doc = name
                break
        
        if not main_doc:
            # htm 파일 중 가장 큰 것 선택
            htm_files = [
                item for item in docs.get("directory", {}).get("item", [])
                if item.get("name", "").endswith(".htm")
            ]
            if htm_files:
                main_doc = max(htm_files, key=lambda x: int(x.get("size", 0))).get("name")
        
        result = {
            "sections": {},
            "financial_data": None,
            "risk_factors": [],
            "business_description": "",
        }
        
        if main_doc:
            content = self.fetch_filing_document_content(cik, accession_number, main_doc)
            soup = BeautifulSoup(content, "html.parser")
            
            # 텍스트 정리
            for tag in soup(["script", "style"]):
                tag.decompose()
            
            text = soup.get_text(separator="\n")
            
            # 10-K 주요 섹션 추출
            sections = self._extract_10k_sections(text)
            result["sections"] = sections
            
            # Risk Factors 추출
            if "Item 1A" in sections:
                result["risk_factors"] = self._parse_risk_factors(sections["Item 1A"])
            
            # Business Description 추출
            if "Item 1" in sections:
                result["business_description"] = sections["Item 1"][:5000]
        
        # XBRL 데이터에서 재무 정보 추출 시도
        financial_data = self._extract_xbrl_financials(cik, accession_number, docs)
        result["financial_data"] = financial_data
        
        return result
    
    def _extract_10k_sections(self, text: str) -> dict:
        """10-K 섹션 추출"""
        sections = {}
        
        # 주요 아이템 패턴
        item_patterns = [
            (r"ITEM\s*1[.\s]+BUSINESS", "Item 1"),
            (r"ITEM\s*1A[.\s]+RISK\s*FACTORS", "Item 1A"),
            (r"ITEM\s*1B[.\s]+UNRESOLVED\s*STAFF\s*COMMENTS", "Item 1B"),
            (r"ITEM\s*2[.\s]+PROPERTIES", "Item 2"),
            (r"ITEM\s*3[.\s]+LEGAL\s*PROCEEDINGS", "Item 3"),
            (r"ITEM\s*4[.\s]+MINE\s*SAFETY", "Item 4"),
            (r"ITEM\s*5[.\s]+MARKET", "Item 5"),
            (r"ITEM\s*6[.\s]+", "Item 6"),
            (r"ITEM\s*7[.\s]+MANAGEMENT.?S\s*DISCUSSION", "Item 7"),
            (r"ITEM\s*7A[.\s]+QUANTITATIVE", "Item 7A"),
            (r"ITEM\s*8[.\s]+FINANCIAL\s*STATEMENTS", "Item 8"),
            (r"ITEM\s*9[.\s]+CHANGES", "Item 9"),
            (r"ITEM\s*9A[.\s]+CONTROLS", "Item 9A"),
            (r"ITEM\s*10[.\s]+DIRECTORS", "Item 10"),
            (r"ITEM\s*11[.\s]+EXECUTIVE\s*COMPENSATION", "Item 11"),
            (r"ITEM\s*12[.\s]+SECURITY\s*OWNERSHIP", "Item 12"),
            (r"ITEM\s*13[.\s]+CERTAIN\s*RELATIONSHIPS", "Item 13"),
            (r"ITEM\s*14[.\s]+PRINCIPAL\s*ACCOUNTANT", "Item 14"),
            (r"ITEM\s*15[.\s]+EXHIBITS", "Item 15"),
        ]
        
        # 각 섹션 위치 찾기
        positions = []
        for pattern, name in item_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                positions.append((match.start(), name))
        
        positions.sort(key=lambda x: x[0])
        
        # 섹션 내용 추출
        for i, (pos, name) in enumerate(positions):
            end_pos = positions[i + 1][0] if i + 1 < len(positions) else len(text)
            section_text = text[pos:end_pos].strip()
            # 처음 10000자만 저장 (너무 길면 잘라냄)
            sections[name] = section_text[:10000]
        
        return sections
    
    def _parse_risk_factors(self, risk_section: str) -> list[str]:
        """Risk Factors 섹션에서 개별 리스크 추출"""
        risks = []
        
        # 일반적인 리스크 헤더 패턴
        # 볼드체나 대문자로 시작하는 문장을 리스크로 간주
        lines = risk_section.split("\n")
        current_risk = []
        
        for line in lines:
            line = line.strip()
            if not line:
                continue
            
            # 새로운 리스크 항목 시작 감지
            if (line.isupper() and len(line) > 20) or \
               (line.endswith(".") and len(line) < 200 and line[0].isupper()):
                if current_risk:
                    risks.append(" ".join(current_risk))
                current_risk = [line]
            else:
                current_risk.append(line)
        
        if current_risk:
            risks.append(" ".join(current_risk))
        
        # 상위 20개 리스크만 반환
        return risks[:20]
    
    def _extract_xbrl_financials(
        self, 
        cik: str, 
        accession_number: str, 
        docs: dict
    ) -> Optional[FinancialData]:
        """XBRL 파일에서 재무 데이터 추출"""
        
        # XBRL JSON 파일 찾기
        xbrl_json = None
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if "Financial" in name and name.endswith(".json"):
                xbrl_json = name
                break
        
        if not xbrl_json:
            return None
        
        try:
            content = self.fetch_filing_document_content(cik, accession_number, xbrl_json)
            data = json.loads(content)
            
            facts = data.get("facts", {})
            us_gaap = facts.get("us-gaap", {})
            
            financial = FinancialData()
            
            # 매출
            revenue_keys = ["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", 
                          "SalesRevenueNet", "RevenueFromContractWithCustomerIncludingAssessedTax"]
            financial.revenue = self._get_latest_value(us_gaap, revenue_keys)
            
            # 순이익
            ni_keys = ["NetIncomeLoss", "ProfitLoss", "NetIncomeLossAvailableToCommonStockholdersBasic"]
            financial.net_income = self._get_latest_value(us_gaap, ni_keys)
            
            # 총자산
            asset_keys = ["Assets"]
            financial.total_assets = self._get_latest_value(us_gaap, asset_keys)
            
            # 총부채
            liability_keys = ["Liabilities", "LiabilitiesAndStockholdersEquity"]
            financial.total_liabilities = self._get_latest_value(us_gaap, liability_keys)
            
            # 자본
            equity_keys = ["StockholdersEquity", "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"]
            financial.stockholders_equity = self._get_latest_value(us_gaap, equity_keys)
            
            # EPS
            eps_keys = ["EarningsPerShareBasic"]
            financial.eps_basic = self._get_latest_value(us_gaap, eps_keys)
            
            eps_diluted_keys = ["EarningsPerShareDiluted"]
            financial.eps_diluted = self._get_latest_value(us_gaap, eps_diluted_keys)
            
            # 현금
            cash_keys = ["CashAndCashEquivalentsAtCarryingValue", "Cash"]
            financial.cash_and_equivalents = self._get_latest_value(us_gaap, cash_keys)
            
            return financial
            
        except Exception as e:
            print(f"XBRL 파싱 오류: {e}")
            return None
    
    def _get_latest_value(self, us_gaap: dict, keys: list) -> Optional[float]:
        """US-GAAP 데이터에서 최신 값 추출"""
        for key in keys:
            if key in us_gaap:
                units = us_gaap[key].get("units", {})
                # USD 값 찾기
                usd_values = units.get("USD", [])
                if usd_values:
                    # 가장 최근 값 (보통 마지막)
                    sorted_values = sorted(
                        usd_values, 
                        key=lambda x: x.get("end", ""), 
                        reverse=True
                    )
                    if sorted_values:
                        return sorted_values[0].get("val")
                
                # USD/shares (EPS용)
                share_values = units.get("USD/shares", [])
                if share_values:
                    sorted_values = sorted(
                        share_values,
                        key=lambda x: x.get("end", ""),
                        reverse=True
                    )
                    if sorted_values:
                        return sorted_values[0].get("val")
        return None
    
    def extract_10q_data(self, cik: str, accession_number: str) -> dict:
        """
        10-Q 파일링에서 주요 데이터 추출
        
        10-Q는 분기 보고서로 10-K와 유사하지만 더 간략함
        """
        docs = self.fetch_filing_documents(accession_number, cik)
        
        result = {
            "sections": {},
            "financial_data": None,
            "md_and_a": "",  # Management Discussion & Analysis
        }
        
        # 메인 문서 찾기
        main_doc = None
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if name.endswith(".htm"):
                doc_type = item.get("type", "")
                if "10-Q" in doc_type or "10-q" in name.lower():
                    main_doc = name
                    break
        
        if not main_doc:
            htm_files = [
                item for item in docs.get("directory", {}).get("item", [])
                if item.get("name", "").endswith(".htm")
            ]
            if htm_files:
                main_doc = max(htm_files, key=lambda x: int(x.get("size", 0))).get("name")
        
        if main_doc:
            content = self.fetch_filing_document_content(cik, accession_number, main_doc)
            soup = BeautifulSoup(content, "html.parser")
            
            for tag in soup(["script", "style"]):
                tag.decompose()
            
            text = soup.get_text(separator="\n")
            
            # 10-Q 섹션 추출
            sections = self._extract_10q_sections(text)
            result["sections"] = sections
            
            # MD&A 추출
            if "Item 2" in sections:
                result["md_and_a"] = sections["Item 2"][:5000]
        
        # 재무 데이터
        financial_data = self._extract_xbrl_financials(cik, accession_number, docs)
        result["financial_data"] = financial_data
        
        return result
    
    def _extract_10q_sections(self, text: str) -> dict:
        """10-Q 섹션 추출"""
        sections = {}
        
        item_patterns = [
            (r"ITEM\s*1[.\s]+FINANCIAL\s*STATEMENTS", "Item 1"),
            (r"ITEM\s*2[.\s]+MANAGEMENT.?S\s*DISCUSSION", "Item 2"),
            (r"ITEM\s*3[.\s]+QUANTITATIVE", "Item 3"),
            (r"ITEM\s*4[.\s]+CONTROLS", "Item 4"),
            (r"PART\s*II", "Part II"),
            (r"ITEM\s*1[.\s]+LEGAL\s*PROCEEDINGS", "Part II Item 1"),
            (r"ITEM\s*1A[.\s]+RISK\s*FACTORS", "Part II Item 1A"),
            (r"ITEM\s*6[.\s]+EXHIBITS", "Part II Item 6"),
        ]
        
        positions = []
        for pattern, name in item_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                positions.append((match.start(), name))
        
        positions.sort(key=lambda x: x[0])
        
        for i, (pos, name) in enumerate(positions):
            end_pos = positions[i + 1][0] if i + 1 < len(positions) else len(text)
            section_text = text[pos:end_pos].strip()
            sections[name] = section_text[:10000]
        
        return sections
    
    def extract_8k_data(self, cik: str, accession_number: str) -> Filing8KData:
        """
        8-K 파일링에서 이벤트 데이터 추출
        
        8-K는 중요한 기업 이벤트 공시
        """
        docs = self.fetch_filing_documents(accession_number, cik)
        
        result = Filing8KData()
        
        # 메인 8-K 문서 찾기
        main_doc = None
        for item in docs.get("directory", {}).get("item", []):
            name = item.get("name", "")
            if name.endswith(".htm") and "8-k" in name.lower():
                main_doc = name
                break
        
        if not main_doc:
            htm_files = [
                item for item in docs.get("directory", {}).get("item", [])
                if item.get("name", "").endswith(".htm")
            ]
            if htm_files:
                main_doc = max(htm_files, key=lambda x: int(x.get("size", 0))).get("name")
        
        if main_doc:
            content = self.fetch_filing_document_content(cik, accession_number, main_doc)
            soup = BeautifulSoup(content, "html.parser")
            
            for tag in soup(["script", "style"]):
                tag.decompose()
            
            text = soup.get_text(separator="\n")
            
            # 8-K 아이템 추출
            items = self._extract_8k_items(text)
            result.items = items
            
            # 각 아이템에 대한 설명
            for item_code in items:
                if item_code in self.ITEM_8K_MAPPING:
                    result.event_descriptions.append({
                        "item": item_code,
                        "description": self.ITEM_8K_MAPPING[item_code],
                        "content": self._extract_item_content(text, item_code)
                    })
        
        return result
    
    def _extract_8k_items(self, text: str) -> list[str]:
        """8-K에서 보고된 아이템 번호 추출"""
        items = []
        
        # Item X.XX 패턴 찾기
        pattern = r"Item\s*(\d+\.\d+)"
        matches = re.findall(pattern, text, re.IGNORECASE)
        
        for match in matches:
            if match in self.ITEM_8K_MAPPING and match not in items:
                items.append(match)
        
        return items
    
    def _extract_item_content(self, text: str, item_code: str) -> str:
        """특정 8-K 아이템의 내용 추출"""
        pattern = rf"Item\s*{re.escape(item_code)}[.\s]+"
        match = re.search(pattern, text, re.IGNORECASE)
        
        if match:
            start = match.end()
            # 다음 Item까지 또는 Signature까지
            end_pattern = r"Item\s*\d+\.\d+|SIGNATURE"
            end_match = re.search(end_pattern, text[start:], re.IGNORECASE)
            end = start + end_match.start() if end_match else start + 5000
            
            content = text[start:end].strip()
            return content[:3000]  # 3000자로 제한
        
        return ""
    
    def search_filings(
        self,
        query: str,
        filing_types: list[str] = None,
        start_date: str = None,
        end_date: str = None,
        size: int = 10
    ) -> list[dict]:
        """
        EDGAR 전문 검색
        
        Args:
            query: 검색어
            filing_types: 파일링 유형 리스트 (예: ["10-K", "8-K"])
            start_date: 시작일 (YYYY-MM-DD)
            end_date: 종료일 (YYYY-MM-DD)
            size: 결과 수
        
        참조:
            https://www.sec.gov/edgar/search/
        """
        url = "https://efts.sec.gov/LATEST/search-index"
        
        params = {
            "q": query,
            "dateRange": "custom",
            "startdt": start_date or "2020-01-01",
            "enddt": end_date or "2025-12-31",
            "forms": ",".join(filing_types) if filing_types else "10-K,10-Q,8-K",
            "from": 0,
            "size": size,
        }
        
        self._rate_limit()
        print(url, params)
        resp = self.session.get(url, params=params)
        resp.raise_for_status()
        
        results = []
        data = resp.json()
        
        for hit in data.get("hits", {}).get("hits", []):
            source = hit.get("_source", {})
            results.append({
                "cik": source.get("ciks", [""])[0],
                "company_name": source.get("display_names", [""])[0],
                "filing_type": source.get("form", ""),
                "filing_date": source.get("file_date", ""),
                "accession_number": source.get("adsh", ""),
            })
        
        return results

In [62]:
# SEC에서 요구하는 User-Agent 설정 (실제 사용시 본인 정보로 변경)
crawler = EDGARCrawler(user_agent="Sayouzone sjkim@sayouzone.com")

def main(ticker: str):
    """사용 예시"""
    cik = crawler.fetch_cik_by_ticker(ticker)
    print(f"\n{ticker} CIK: {cik}")
    
    # 10-K 파일링 목록 조회
    print("\n=== 최근 10-K 파일링 ===")
    filings_10k = crawler.fetch_company_filings(cik, "10-K", count=3)
    for f in filings_10k:
        print(f"  {f.filing_date}: {f.accession_number}")
    
    # 가장 최근 10-K 데이터 추출
    if filings_10k:
        print("\n=== 최근 10-K 데이터 추출 ===")
        latest_10k = filings_10k[0]
        data_10k = crawler.extract_10k_data(cik, latest_10k.accession_number)
        
        print(f"추출된 섹션: {list(data_10k['sections'].keys())}")
        
        if data_10k["financial_data"]:
            fd = data_10k["financial_data"]
            print(f"\n재무 데이터:")
            if fd.revenue:
                print(f"  매출: ${fd.revenue:,.0f}")
            if fd.net_income:
                print(f"  순이익: ${fd.net_income:,.0f}")
            if fd.total_assets:
                print(f"  총자산: ${fd.total_assets:,.0f}")
            if fd.eps_diluted:
                print(f"  희석 EPS: ${fd.eps_diluted:.2f}")
        
        if data_10k["risk_factors"]:
            print(f"\n리스크 팩터 수: {len(data_10k['risk_factors'])}")
            print(f"첫 번째 리스크: {data_10k['risk_factors'][0][:200]}...")
    
    # 10-Q 파일링 조회
    print("\n=== 최근 10-Q 파일링 ===")
    filings_10q = crawler.fetch_company_filings(cik, "10-Q", count=3)
    for f in filings_10q:
        print(f"  {f.filing_date}: {f.accession_number}")
    
    # 8-K 파일링 조회 및 추출
    print("\n=== 최근 8-K 파일링 ===")
    filings_8k = crawler.fetch_company_filings(cik, "8-K", count=5)
    for f in filings_8k:
        print(f"  {f.filing_date}: {f.accession_number}")
    
    if filings_8k:
        print("\n=== 최근 8-K 이벤트 분석 ===")
        latest_8k = filings_8k[0]
        data_8k = crawler.extract_8k_data(cik, latest_8k.accession_number)
        
        print(f"보고된 아이템: {data_8k.items}")
        for event in data_8k.event_descriptions:
            print(f"\n  Item {event['item']}: {event['description']}")
            if event['content']:
                print(f"    내용 미리보기: {event['content'][:200]}...")
    
    test_13f()
    test_def14()

In [ ]:
def test_13f():
    # ========================================
    # 13F 기관투자자 파일링 예시
    # ========================================
    print("\n" + "="*60)
    print("13F 기관투자자 포트폴리오 분석")
    print("="*60)
    
    # 예시: Berkshire Hathaway (버크셔 해서웨이)
    # Warren Buffett의 투자 포트폴리오
    cik = "0001067983"

    edgar_13f = EDGAR13F()
    
    print(f"\nBerkshire Hathaway CIK: {cik}")
    
    # 13F 파일링 목록 조회
    print("\n=== 최근 13F-HR 파일링 ===")
    filings_13f = edgar_13f.fetch_13f_filings(cik, count=4)
    for f in filings_13f:
        print(f"  {f.filing_date}: {f.accession_number}")
    
    if filings_13f:
        # 가장 최근 13F 데이터 추출
        print("\n=== 최근 13F 포트폴리오 분석 ===")
        latest_13f = filings_13f[0]
        data_13f = edgar_13f.extract_13f_data(cik, latest_13f.accession_number)
        
        print(f"기관명: {data_13f.filer_name}")
        print(f"보고 기간: {data_13f.report_period}")
        print(f"총 운용자산: ${data_13f.total_value / 1000:,.2f}M")
        print(f"보유 종목 수: {data_13f.holdings_count}")
        
        print("\n상위 10개 보유 종목:")
        for i, h in enumerate(data_13f.top_holdings, 1):
            print(f"  {i}. {h['issuer']}")
            print(f"     CUSIP: {h['cusip']}")
            print(f"     가치: ${h['value_millions']:,.2f}M ({h['percentage']:.2f}%)")
            print(f"     주식 수: {h['shares']:,.0f}")
        
        # 두 분기 비교 (파일링이 2개 이상인 경우)
        if len(filings_13f) >= 2:
            print("\n=== 분기별 포트폴리오 변화 분석 ===")
            comparison = extractor.compare_13f_filings(
                cik,
                filings_13f[0].accession_number,
                filings_13f[1].accession_number
            )
            
            print(f"비교 기간: {comparison['previous_period']} → {comparison['current_period']}")
            print(f"총 자산 변화: ${comparison['total_value_change'] / 1000:,.2f}M")
            print(f"종목 수 변화: {comparison['previous_holdings_count']} → {comparison['current_holdings_count']}")
            
            if comparison["new_positions"]:
                print("\n신규 매수 (상위 5개):")
                for p in comparison["new_positions"][:5]:
                    print(f"  - {p['issuer']}: ${p['value_thousands'] / 1000:,.2f}M")
            
            if comparison["closed_positions"]:
                print("\n완전 매도 (상위 5개):")
                for p in comparison["closed_positions"][:5]:
                    print(f"  - {p['issuer']}: ${p['previous_value_thousands'] / 1000:,.2f}M")
            
            if comparison["increased_positions"]:
                print("\n비중 확대 (상위 5개):")
                for p in comparison["increased_positions"][:5]:
                    print(f"  - {p['issuer']}: {p['share_change_pct']:+.1f}% ({p['share_change']:+,.0f}주)")
            
            if comparison["decreased_positions"]:
                print("\n비중 축소 (상위 5개):")
                for p in comparison["decreased_positions"][:5]:
                    print(f"  - {p['issuer']}: {p['share_change_pct']:+.1f}% ({p['share_change']:+,.0f}주)")
    
    # 다른 유명 기관투자자 예시
    print("\n=== 다른 기관투자자 13F 예시 ===")
    famous_investors = {
        "Bridgewater Associates": "0001350694",
        "Renaissance Technologies": "0001037389", 
        "Citadel Advisors": "0001423053",
        "Two Sigma Investments": "0001179392",
    }
    
    for name, investor_cik in famous_investors.items():
        print(f"\n{name} (CIK: {investor_cik})")
        try:
            filings = extractor.fetch_13f_filings(investor_cik, count=1)
            if filings:
                data = extractor.extract_13f_data(investor_cik, filings[0].accession_number)
                print(f"  보고 기간: {data.report_period}")
                print(f"  총 운용자산: ${data.total_value / 1000:,.2f}M")
                print(f"  보유 종목 수: {data.holdings_count}")
                if data.top_holdings:
                    print(f"  최대 보유: {data.top_holdings[0]['issuer']}")
        except Exception as e:
            print(f"  데이터 조회 실패: {e}")

In [ ]:
def test_def14():
    # ========================================
    # DEF 14A Proxy Statement 예시
    # ========================================
    print("\n" + "="*60)
    print("DEF 14A Proxy Statement 분석")
    print("="*60)
    
    # Apple의 DEF 14A 조회
    print(f"\n{ticker} DEF 14A 분석")
    
    # DEF 14A 파일링 목록 조회
    print("\n=== 최근 DEF 14A 파일링 ===")
    filings_def14a = extractor.get_def14a_filings(cik, count=3)
    for f in filings_def14a:
        print(f"  {f.filing_date}: {f.accession_number}")
    
    if filings_def14a:
        print("\n=== 최근 DEF 14A 데이터 추출 ===")
        latest_proxy = filings_def14a[0]
        proxy_data = extractor.extract_def14a_data(cik, latest_proxy.accession_number)
        
        # 주주총회 정보
        print(f"\n주주총회 정보:")
        print(f"  회의 유형: {proxy_data.meeting_type}")
        print(f"  회의 날짜: {proxy_data.meeting_date}")
        print(f"  기준일: {proxy_data.record_date}")
        
        # 임원 보상
        if proxy_data.executive_compensation:
            print(f"\n임원 보상 (Named Executive Officers):")
            for exec_comp in proxy_data.executive_compensation[:5]:
                print(f"\n  {exec_comp.name}")
                if exec_comp.title:
                    print(f"    직책: {exec_comp.title}")
                print(f"    기본급: ${exec_comp.salary:,.0f}")
                print(f"    보너스: ${exec_comp.bonus:,.0f}")
                print(f"    주식 보상: ${exec_comp.stock_awards:,.0f}")
                print(f"    옵션 보상: ${exec_comp.option_awards:,.0f}")
                print(f"    총 보상: ${exec_comp.total:,.0f}")
        
        # CEO Pay Ratio
        if proxy_data.ceo_pay_ratio:
            print(f"\nCEO Pay Ratio:")
            print(f"  CEO: {proxy_data.ceo_name}")
            print(f"  Pay Ratio: {proxy_data.ceo_pay_ratio:.0f}:1")
            if proxy_data.median_employee_pay:
                print(f"  중간 직원 급여: ${proxy_data.median_employee_pay:,.0f}")
        
        # 이사회
        if proxy_data.directors:
            print(f"\n이사회 구성:")
            print(f"  이사 수: {proxy_data.board_size}")
            print(f"  독립 이사: {proxy_data.independent_directors}")
            print(f"\n  이사 명단:")
            for director in proxy_data.directors[:10]:
                age_str = f", {director.age}세" if director.age else ""
                ind_str = " (Independent)" if director.independent else ""
                print(f"    - {director.name}{age_str}{ind_str}")
        
        # 투표 안건
        if proxy_data.proposals:
            print(f"\n투표 안건:")
            for prop in proxy_data.proposals:
                rec = f" [Board: {prop.board_recommendation}]" if prop.board_recommendation else ""
                print(f"  {prop.proposal_number}. {prop.title}{rec}")
        
        # 주요 주주
        if proxy_data.major_shareholders:
            print(f"\n주요 주주:")
            for sh in proxy_data.major_shareholders[:10]:
                type_str = "[기관]" if sh.is_institution else "[내부자]"
                print(f"  {type_str} {sh.name}: {sh.percentage:.2f}%")
        
        # 거버넌스
        if proxy_data.governance_highlights:
            print(f"\n거버넌스 특징:")
            for item in proxy_data.governance_highlights:
                print(f"  ✓ {item}")
        
        if proxy_data.say_on_pay_vote:
            print(f"\nSay on Pay: {proxy_data.say_on_pay_vote}")
            if proxy_data.prior_say_on_pay_support:
                print(f"  작년 지지율: {proxy_data.prior_say_on_pay_support:.1f}%")
        
        # 임원 보상 트렌드 분석
        print("\n=== 임원 보상 트렌드 분석 (최근 3년) ===")
        try:
            trends = extractor.analyze_executive_compensation_trends(cik, years=3)
            for year_data in trends["compensation_by_year"]:
                print(f"\n  {year_data['filing_date']}:")
                print(f"    CEO: {year_data['ceo_name']}")
                if year_data.get('ceo_total_compensation'):
                    print(f"    CEO 총 보상: ${year_data['ceo_total_compensation']:,.0f}")
                if year_data.get('ceo_comp_yoy_change_pct'):
                    print(f"    전년 대비: {year_data['ceo_comp_yoy_change_pct']:+.1f}%")
                if year_data.get('ceo_pay_ratio'):
                    print(f"    Pay Ratio: {year_data['ceo_pay_ratio']:.0f}:1")
        except Exception as e:
            print(f"  트렌드 분석 실패: {e}")
    
    # 동종 업계 비교 예시
    print("\n=== 동종 업계 임원 보상 비교 ===")
    tech_companies = {
        "Apple": "0000320193",
        "Microsoft": "0000789019",
        "Alphabet": "0001652044",
        "Meta": "0001326801",
        "Amazon": "0001018724",
    }
    
    print("Big Tech CEO 보상 비교:")
    try:
        comparison = extractor.compare_peer_compensation(list(tech_companies.values()))
        for comp in comparison:
            if comp.get("ceo_total_compensation"):
                print(f"\n  {comp.get('ceo_name', 'N/A')} ({comp['cik']})")
                print(f"    총 보상: ${comp['ceo_total_compensation']:,.0f}")
                if comp.get('ceo_pay_ratio'):
                    print(f"    Pay Ratio: {comp['ceo_pay_ratio']:.0f}:1")
    except Exception as e:
        print(f"  비교 분석 실패: {e}")

In [67]:
def search(query: str):
    """
    dateRange: all (All (since 2001), 10y (Last 10 years), 5y (Last 5 years), 1y (Last year), 30d (Last 30 days), custom (Custom)
    ciks: CIK number
    entityName: Apple%20Inc.%20(AAPL)%20(CIK%200000320193)
    startdt: Filed from
    enddt: Filed to
    """
    print("\n=== EDGAR 검색: 'artificial intelligence' ===")
    search_results = crawler.search_filings(
        query=query,
        filing_types=["10-K", "8-K"],
        start_date="2025-01-01",
        size=5
    )
    for r in search_results:
        print(f"  {r['company_name']}: {r['filing_type']} ({r['filing_date']})")

https://efts.sec.gov/LATEST/search-index?q=HBM&ciks=0000320193&entityName=&startdt=2020-12-13&enddt=2025-12-13

https://efts.sec.gov/LATEST/search-index 
{'q': 'HBM', 'dateRange': 'custom', 'startdt': '2025-01-01', 'enddt': '2025-12-31', 'from': 0, 'size': 5}
{'q': 'HBM', 'dateRange': 'custom', 'startdt': '2025-01-01', 'enddt': '2025-12-31', 'forms': '10-K,8-K', 'from': 0, 'size': 5}

## Test

In [68]:
# 예시: Apple (AAPL)
#ticker = "AAPL"
ticker = "GOOG"
main(ticker)


GOOG CIK: 0001652044

=== 최근 10-K 파일링 ===
  2025-02-05: 0001652044-25-000014
  2024-01-31: 0001652044-24-000022
  2023-02-03: 0001652044-23-000016
  2022-02-02: 0001652044-22-000019
  2021-02-03: 0001652044-21-000010
  2020-02-04: 0001652044-20-000008
  2019-02-06: 0001193125-19-028757
  2019-02-05: 0001652044-19-000004
  2018-02-06: 0001652044-18-000007
  2017-02-03: 0001652044-17-000008

=== 최근 10-K 데이터 추출 ===
추출된 섹션: ['Item 1', 'Item 1A', 'Item 1B', 'Item 2', 'Item 3', 'Item 4', 'Item 5', 'Item 6', 'Item 7', 'Item 7A', 'Item 8', 'Item 9', 'Item 9A', 'Item 10', 'Item 11', 'Item 12', 'Item 13', 'Item 14', 'Item 15']

리스크 팩터 수: 1
첫 번째 리스크: Item 1A. Risk Factors 10...

=== 최근 10-Q 파일링 ===
  2025-10-30: 0001652044-25-000091
  2025-07-24: 0001652044-25-000062
  2025-04-25: 0001652044-25-000043
  2024-10-30: 0001652044-24-000118
  2024-07-24: 0001652044-24-000079
  2024-04-26: 0001652044-24-000053
  2023-10-25: 0001652044-23-000094
  2023-07-26: 0001652044-23-000070
  2023-04-26: 00016520

In [46]:
#query = "artificial intelligence"
query = "HBM"
search(query)


=== EDGAR 검색: 'artificial intelligence' ===
https://efts.sec.gov/LATEST/search-index {'q': 'HBM', 'dateRange': 'custom', 'startdt': '2025-01-01', 'enddt': '2025-12-31', 'forms': '10-K,8-K', 'from': 0, 'size': 5}
  SPRUCE BIOSCIENCES, INC.  (SPRB)  (CIK 0001683553): 10-K (2025-04-15)
  MICRON TECHNOLOGY INC  (MU)  (CIK 0000723125): 8-K (2025-06-12)
  VEECO INSTRUMENTS INC  (VECO)  (CIK 0000103145): 8-K (2025-05-07)
  VEECO INSTRUMENTS INC  (VECO)  (CIK 0000103145): 8-K (2025-02-12)
  FORMFACTOR INC  (FORM)  (CIK 0001039399): 8-K (2025-07-30)
  CANADIAN DERIVATIVES CLEARING CORP  (CIK 0000319643): 8-K (2025-06-09)
  CANADIAN DERIVATIVES CLEARING CORP  (CIK 0000319643): 8-K (2025-03-10)
  MICRON TECHNOLOGY INC  (MU)  (CIK 0000723125): 10-K (2025-10-03)
  CANADIAN DERIVATIVES CLEARING CORP  (CIK 0000319643): 8-K (2025-07-09)
  CANADIAN DERIVATIVES CLEARING CORP  (CIK 0000319643): 8-K (2025-12-02)
  FORMFACTOR INC  (FORM)  (CIK 0001039399): 8-K (2025-02-05)
  CANADIAN DERIVATIVES CLEARING 

{"error":"Blank search not valid.  Either entity, keywords, location, or filing types must be submitted.","hits":{"hits":[]}}

- https://efts.sec.gov/LATEST/search-index?q=artificial%20intelligence&dateRange=custom
- https://efts.sec.gov/LATEST/search-index?q=artificial%20intelligence&dateRange=custom&startdt=2024-01-01&enddt=2025-12-31
- https://efts.sec.gov/LATEST/search-index?q=artificial%20intelligence&dateRange=custom&startdt=2024-01-01&enddt=2025-12-31&forms=10-K,8-K

In [76]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# SEC EDGAR Base URL
SEC_BASE_URL = "https://www.sec.gov/Archives"

# Fetch Form 13F Filings for a Given CIK
def fetch_13f_filings(cik, num_filings=2):
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=13F&count={num_filings}&output=atom"
    headers = {"User-Agent": "Sayouzone (sjkim@sayouzone.com)"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "xml")
    entries = soup.find_all("entry")
    filings = [(entry.find("filing-date").text, entry.find("link")["href"]) for entry in entries]
    return filings[:num_filings]

# Parse Holdings from XML Data
def parse_13f_holdings(filing_url):
    headers = {"User-Agent": "Sayouzone (sjkim@sayouzone.com)"}
    response = requests.get(filing_url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "xml")
    rows = []
    for info_table in soup.find_all("infoTable"):
        name = info_table.find("nameOfIssuer").text
        ticker = info_table.find("cusip").text
        value = int(info_table.find("value").text) * 1000  # Value in dollars
        shares = int(info_table.find("sshPrnamt").text)
        rows.append({"Name": name, "Ticker": ticker, "Value": value, "Shares": shares})
    return pd.DataFrame(rows)

# Compare Holdings Between Two Quarters
def compare_holdings(holdings_q1, holdings_q2):
    merged = holdings_q1.merge(holdings_q2, on="Ticker", how="outer", suffixes=("_q1", "_q2"))
    merged["Value_Change"] = merged["Value_q2"].fillna(0) - merged["Value_q1"].fillna(0)
    merged["Shares_Change"] = merged["Shares_q2"].fillna(0) - merged["Shares_q1"].fillna(0)
    return merged

# Main Function
def main():
    blackrock_cik = "0001364742"  # CIK for BlackRock
    filings = fetch_13f_filings(blackrock_cik)
    
    all_holdings = []
    for date, filing_url in filings:
        print(f"Processing filing from {filing_url} {date}...")
        #filing_content = filing_url.replace("-index.htm", ".xml")
        """
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/0001652044-25-000096-index.htm
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/xslForm13F_X02/primary_doc.xml
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/primary_doc.xml
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/xslForm13F_X02/information_table.xml
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/information_table.xml
        https://www.sec.gov/Archives/edgar/data/1652044/000165204425000096/0001652044-25-000096.txt
        """
        #holdings.append(parse_13f_holdings(filing_content))
        #holdings.append(parse_13f_holdings(filing_url))
        holdings = parse_13f_holdings(filing_url)
        print(holdings)
        all_holdings.append(holdings)

    if len(holdings) == 2:
        comparison = compare_holdings(holdings[0], holdings[1])
        comparison.to_csv("blackrock_holdings_comparison.csv", index=False)
        print("Holdings comparison saved to 'blackrock_holdings_comparison.csv'")

main()

Processing filing from https://www.sec.gov/Archives/edgar/data/1364742/000108636424008417/0001086364-24-008417-index.htm 2024-08-13...
Empty DataFrame
Columns: []
Index: []
Processing filing from https://www.sec.gov/Archives/edgar/data/1364742/000108636424007638/0001086364-24-007638-index.htm 2024-05-10...
Empty DataFrame
Columns: []
Index: []
